# Análise da Campanha Mínima Viável

Métricas por réplica, critérios de aceitação (Tabela 4.6 da v0.8) e comparações pareadas entre MECADE e baseline.

Para a campanha real, `DATA_ROOT` aponta para `data/runs`. Uso `data/sim/runs` (com `INCLUDE_SIM=True`) só para testar o pipeline, e esses dados simulados não entram em nenhum resultado.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'analysis' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from analysis.metrics import audit_overhead, execution_failures, false_block_verdict, load_official_runs, load_runs, metrics_table, paired_comparison
DATA_ROOT = ROOT / 'data' / 'runs'
INCLUDE_SIM = False
FIG = ROOT / 'analysis' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
if INCLUDE_SIM:
    runs, excluidas = load_runs(DATA_ROOT, include_sim=True), []
else:  # análise oficial: só campaign_id do config, sem sim nem calibração
    runs, excluidas = load_official_runs(DATA_ROOT)
print(len(excluidas), 'réplicas excluídas:', excluidas)
m = metrics_table(runs)
print(len(runs), 'réplicas carregadas'); m.head()

## 1. Visão geral por cenário e braço

In [ ]:
cols = ['detected','TTD','TTM','TTR_capped','MTTR','area_incidente','p95_std','p99_std','TSR_fault','p99_fault','RRIndex_run']
main = m[m['pair'].astype(str) != '']
main.groupby(['scenario','arm'])[cols].median().round(3)

## 2. Classificação tripartite (Tabela 3.5) e detecção

In [ ]:
pd.crosstab([main['scenario'], main['arm']], main['classe'])

In [ ]:
main.groupby(['scenario','arm']).agg(n=('run_id','count'), detectadas=('detected','sum'), censuradas=('censurada','sum'), rto_ok=('rto_ok','sum'))

In [ ]:
falhas = execution_failures(DATA_ROOT)
print('falha_execucao (o par sai da comparação):', [(f['run_id'], f['pair']) for f in falhas])
print('censuradas:', list(main[main['censurada']]['run_id']))

## 3. Comparações pareadas (MECADE menos baseline)

Wilcoxon pareado, delta de Cliff e IC de 95% por *bootstrap* da mediana das diferenças. Réplicas censuradas entram com o fim da janela (`_capped`), como manda a Seção 4.4.1 da v0.8.

In [ ]:
pd.concat([paired_comparison(m, k) for k in ['TTD_capped','TTR_capped','MTTR_capped','area_incidente','p95_std','p99_std','RRIndex_run','TSR_fault','p99_fault']], ignore_index=True).round(4)

## 4. Taxa de falsos bloqueios (réplicas de controle)

In [ ]:
from mecade.config import load_campaign
v = false_block_verdict(m, load_campaign()['analysis']['false_block_max_rate'])
print(f"Controles: {v['controles']} | BLOCK acionado em {v['blocks']} => taxa de falsos bloqueios = {v['taxa']:.2%}")
print(f"Critério: taxa <= {v['limiar']:.0%} (no máximo 1 em 10) ->", v['aceito'])

## 5. Overhead da auditoria (Camada 6, perfil mínimo)

Δ_audit no caminho de decisão é o p95 do custo de enfileirar o evento. Δcpu compara a CPU do controlador nos controles com auditoria ligada e desligada.

In [ ]:
aud = m[m['arm']=='mecade'].groupby('audit')[['audit_p95_ms','controller_cpu_s']].median()
print(aud)
o = audit_overhead(m); print(o)
print('Critério: Δ_audit,p95 <= 5 ms ->', o['delta_audit_p95_ms'] <= 5, '| Δcpu <= 8% ->', o['delta_cpu_pct'] <= 8)

## 6. Vereditos (decisão D1, 2026-11-12)

Critério por cenário: mediana da diferença pareada do lado favorável ao MECADE, IC 95% *bootstrap* sem cruzar zero e limiar da Tabela 4.6 cumprido. Hipótese: ACEITA (todos os critérios em todos os cenários), PARCIAL ou NÃO ACEITA. O mapa de critério para hipótese está em `analysis.veredito` de `config/campaign.yaml`. Componente marcada como pendente não decide, e a hipótese afetada sai PENDENTE. A análise principal não tem correção; Holm é a secundária.

In [ ]:
from analysis.veredito import resumo
cfg = load_campaign()['analysis']
tab_v, crit, hip, crit_holm, hip_holm = resumo(m, cfg['veredito'], cfg)
print('Análise principal, sem correção')
crit

In [ ]:
hip

In [ ]:
print('Análise secundária, com Holm (família única das componentes pareadas decisivas)')
crit_holm

In [ ]:
hip_holm

In [ ]:
tab_v.round(4)

## 7. Figuras

In [ ]:
def fig_series(scenario):
    fig, ax = plt.subplots(figsize=(7, 3.6), dpi=150)
    for meta, df in runs:
        if meta['scenario'] != scenario or not meta.get('pair'):
            continue
        d = df[df['t'] >= 0]
        ax.plot(d['t'], d['D_agr'], lw=0.8, alpha=0.6, color='tab:red' if meta['arm']=='mecade' else 'tab:gray')
    ax.axvline(0, ls=':', c='k'); ax.set_xlabel('t desde a injeção (s)'); ax.set_ylabel('D_agr')
    ax.set_title(f'{scenario}: MECADE (vermelho) × baseline (cinza)')
    fig.tight_layout(); fig.savefig(FIG / f'Dagr_{scenario}.png', dpi=300)
for sc in sorted(main['scenario'].unique()):
    fig_series(sc)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6), dpi=150)
main.boxplot(column='TTR_capped', by=['scenario','arm'], ax=ax, rot=30)
ax.set_ylabel('TTR (s)'); plt.suptitle(''); fig.tight_layout(); fig.savefig(FIG / 'TTR_box.png', dpi=300)

## 8. Exportação para a dissertação

In [ ]:
m.to_csv(ROOT / 'analysis' / 'metricas_por_replica.csv', index=False)
print('analysis/metricas_por_replica.csv gravado')